In [33]:
print(0)

0


### Diffusion Model 

The model simply adds noise to to the sample at different time-steps that are chosen randomly, and then during the denoising step, the noise is predicted. The loss is calculated and used during optimization to update the weights. 

- $x_{o}$ = original sample
- $x_{t}$ = noisy sample
- y<sub>t</sub> = noise at time-step t
- y' = predicted noise
- $\epsilon$ = error
- T = noise scale
- $\alpha$ = 
- $\bar{\alpha}$

x<sub>o</sub>  --> [add noise] x<sub>o</sub> +  y<sub>t</sub> -->  [predict the noise] y' -->  e = y<sub>t</sub> - y'

The model can be made more effiecient placing it in the latent space of an autoencoder.

In [34]:
from pathlib import Path

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

In [ ]:
rgb_size: int = 3
hidden_size: int = 32
embedding_size: int = 64  
batch: int = 3

The size of the sample is reduced by the encoder, but the number of channels is increased to increase the dimension of the latent space or which also acts like an embedding. The dimension chosen for the hidden space is 64

A VAE is used to reduce the features by extracting only import features, but to also expand the features that are extracted

In [ ]:
class Encoder (nn.Module):

    def __init__(self, embedding_size, rgb_size, hidden_size):
        self.__super__()

        self.mu = nn.Sequential(
            #  input: batch x 3 x W x H
            # output: batch x 32 x W x H
            nn.Conv2d(rgb_size, hidden_size, kernel_size=3, stride=4),
            nn.ReLU(),
            #  input: batch x 32 x W x H
            # output: batch x 64 x W x H        ``
            nn.Conv2d(hidden_size, embedding_size, kernel_size=3)
        )

        self.log_var = nn.Sequential(
                    nn.Conv2d(rgb_size, hidden_size, kernel_size=3, stride=4),
                    nn.ReLU(),
                    nn.Conv2d(hidden_size, embedding_size, kernel_size=3, stride=2)
                )


    def forward(self, x: Tensor):
        return self.mu(x), self.log_var(x)

The VAE's decoder is simply the reverse of the encoder, it does not have any activation function because the sample is not mormalized


In [ ]:
class Decoder (nn.Module):

    def __init__(self, embedding_size, rgb_size, hidden_size):
        self.__super__()

        self.model = nn.Sequential(
            #  input: batch x 64 x W x H
            # output: batch x 32 x W x H
            nn.ConvTranspose2d(embedding_size, hidden_size, kernel_size=3, stride=2),
            nn.ReLU(),
            #  input: bactch x 32 x W x H
            # output: bactch x 3 X W x H
            nn.ConvTranspose2d(hidden_size, rgb_size, kernel_size=3, stride=4)
        )

    def forward(self, z: Tensor):
        return self.model(z)

Diffusion Models require a time embedding

In [ ]:
class TimeEmbedding (nn.Module):

    def __init__(self):
        self.__super__()

        self.model = nn.Sequential(
            nn.Linear(1, embedding_size),
            # Sigmoid Linear Unit
            nn.SiLU(),
            nn.Linear(embedding_size, embedding_size)
        )

    def forward(self, t):
        # Tensor = 64 x 64
        embedding: Tensor = self.model(t)
        # add one more dimension to shape
        return embedding[:]

A generic denoiser, specifically a U-Net was chosen for this project, until it is customized in the next project

In [ ]:
class SimpleUNet (nn.Module):

    def __init__(self, T: int, 
                 rgb_size: Tensor,
                 hidden_size: Tensor,
                 embedding_size: Tensor):
        
        self.__super__()

        beta = torch.linspace(
            0.0001,
            0.02,
            T
        )
        alpha = 1 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        self.alpha_bar = alpha_bar[:, None, None, None]

        self.time_model = TimeEmbedding()

        self.downsample1 = nn.Conv2d(rgb_size, hidden_size)
        self.downsample1sample2 = nn.Conv2d(hidden_size, embedding_size)
        self.bottleneck = nn.Conv2d(embedding_size, embedding_size)
        self.upsample = nn.ConvTranspose2d(embedding_size, hidden_size)
        # used to adjust the dimension
        self.projection = nn.Conv2d(hidden_size, hidden_size)
        self.terminal = nn.Conv2d(hidden_size, rgb_size)


    def forward(self, z0: Tensor):
        # noise
        epsilon = torch.randn_like(z0)

        # noisy latent representation
        zt =  (
            (torch.exp(self.alpha_bar) * z0) +
            (torch.exp(1 - self.alpha_bar) * epsilon)
        )

        # up sampling

        # shape: batch x hidden x W x H
        h1 = F.relu(self.downsample1(zt))    

        # shape: batch x emddedding x W x H
        h2 = F.relu(self.downsample2(h1))

        # bottleneck
        # shape: batch x embedding x W x H
        h2 = F.relu(self.bottleneck(h2))

        # down sampling
        # shape: batch x hidden x W x H
        h2 = self.upsample(h2)

        # skip connection
        # shape: batch x  (2 * hidden) x W x H
        h3 = torch.cat([h2, h1], dim=1)

        # projection
        # shape: batch x hidden x W x H
        h3 = F.relu(self.projection(h3))

        out = self.terminal(h3)









All the classes defined are brought together in this class, to organize the sequence of their operation

In [ ]:
class Diffusion(nn.Module):

    def __init__(self):
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.time_model = TimeEmbedding()
        

    def latent_repr(self, mu: Tensor, logvar):
        '''Latent Representation'''
        sigma = torch.exp(0.5 * logvar)  # standard deviation
        epsilon: Tensor = torch.randn_like(sigma)
        return mu + (sigma * epsilon)

    def forward(self, x: Tensor, T: int):
        #-----------------
        # down sampling
        #-----------------
        mu, logvar = self.encoder(x)

        #-------------
        # Bottleneck
        #-------------
        z = self.latent_repr(mu, logvar)

        # diffusion iteration
        # Diffusion Horizon or the total number of time-steps T 

        # random noise with dimension: [batch x T]  2-D Tensor
        # each row contains an independent time-step for each sample in the batchsize
        # if batch is 3 and T is 2: (2 x 3)
        #                             [ [1, 0, 2],
        #                               [2, 1, 0]  
        #                             ]
        #
        noise: Tensor = torch.rand(batch, T)
        # row independent permutation of numbers from 0 to T-1
        perm: Tensor = torch.argsort(noise, dim=1)

        denoiser = SimpleUNet(T, rgb_size, hidden_size, embedding_size)
        
        for t in perm:
            # each row of the 2 dimensional tensor is passed as t
            # t is a vector of length = batch size

            t = t.float()
            # reshape the dimension of the tensor into a (batch x 1) 
            # so it now has 4 rows and 1 column
            t = t[:, None]
            t_emb: Tensor = self.time_model(t)

            # the latent space has a shape of (batch x embedding_size x W x H)
            # t_emb at this time is (batch x embedding_size)
            # {batch x embedding_size} --> {batch x embedding_size x 1 x 1}
            t_emb = t_emb[:, :, None, None]

            # time-conditioned latent represetation
            zt = z + t_emb 

            pred = denoiser(zt)





        # up sampling
